# Chunking: Facts and objectives 

Goal of this notebook:
- load the reconstructed saved HotpotQA corpus (See previous NoteBook 02)
- split documents into chunks of fixed token size
- prepare two corpus variants:
  - chunk size = 128
  - chunk size = 256 

Notes:
- chunking is applied after corpus construction
- chunk size is measured in tokens, not characters
- these chunks will later be embedded and indexed in FAISS

In [1]:
# Load Corpus 
import json
from pathlib import Path

DATA_DIR = Path("/home/a/arfaoui/rag_project/data")
corpus_path = DATA_DIR / "hotpotqa_corpus_500.json"

with open(corpus_path, "r", encoding="utf-8") as f:
    corpus = json.load(f)
# Print to check 
print("Loaded corpus size:", len(corpus))
print("Example document:")
print(corpus[0])

Loaded corpus size: 4962
Example document:
{'title': 'Wedding Dress (film)', 'text': 'Wedding Dress is a South Korean drama film, released on January 14, 2010.'}


In [2]:
# We load a tokenizer to measure chunk size in TOKENS, not characters.
# Using a tokenizer makes chunking consistent and reproducible.


from transformers import AutoTokenizer

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# print to check
print("Tokenizer loaded successfully.")
print("Tokenizer name:", MODEL_NAME)

/home/a/arfaoui/miniconda3/envs/AAML/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer loaded successfully.
Tokenizer name: meta-llama/Llama-3.2-1B-Instruct


In [3]:
# Quick sanity check on the tokenizer
test_text = corpus[0]["text"]
tokens = tokenizer.encode(test_text)

print(f"Test document title: {corpus[0]['title']}")
print(f"Test document text: {test_text}")
print(f"Character count: {len(test_text)}")
print(f"Token count: {len(tokens)}")
print(f"Ratio chars/tokens: {len(test_text)/len(tokens):.2f}")

Test document title: Wedding Dress (film)
Test document text: Wedding Dress is a South Korean drama film, released on January 14, 2010.
Character count: 73
Token count: 21
Ratio chars/tokens: 3.48


In [4]:
# Check token length distribution of corpus documents
# Before blindly applying chunk sizes from the first draft discussed with prof Luckow (128, 256),
# I first inspect the actual token length distribution of my corpus.
# This is important because:
# - Chunk sizes only matter if they actually split documents.
# - A chunk_size larger than most documents means no real chunking happens.
# - Choosing chunk sizes without this check leads to meaningless comparisons.
# - This analysis lets us decide chunk sizes RATIONALLY based on real data.
import numpy as np

token_lengths = [
    len(tokenizer.encode(doc["text"])) 
    for doc in corpus
]

print(f"Min tokens:    {np.min(token_lengths)}")
print(f"Max tokens:    {np.max(token_lengths)}")
print(f"Mean tokens:   {np.mean(token_lengths):.1f}")
print(f"Median tokens: {np.median(token_lengths):.1f}")
print(f"75th pct:      {np.percentile(token_lengths, 75):.1f}")
print(f"90th pct:      {np.percentile(token_lengths, 90):.1f}")
print(f"95th pct:      {np.percentile(token_lengths, 95):.1f}")

# How many docs would actually get split?
for size in [32, 64, 128, 256]:
    n_split = sum(1 for l in token_lengths if l > size)
    print(f"chunk_size={size} → {n_split}/{len(corpus)} docs would be split ({100*n_split/len(corpus):.1f}%)")

Min tokens:    12
Max tokens:    1397
Mean tokens:   128.6
Median tokens: 115.0
75th pct:      161.0
90th pct:      216.0
95th pct:      259.0
chunk_size=32 → 4858/4962 docs would be split (97.9%)
chunk_size=64 → 4186/4962 docs would be split (84.4%)
chunk_size=128 → 2041/4962 docs would be split (41.1%)
chunk_size=256 → 263/4962 docs would be split (5.3%)


### Observations  Token Length Distribution & Chunk Size Selection
chunk_size=256 splits only 5.3% of documents: almost no real chunking occurs,
making it a poor experimental choice on its own but useful as an upper bound
to observe the effect of very large context windows.

chunk_size=128 splits 41.1% of documents: a meaningful middle ground that
represents the standard default in RAG literature and covers the median document length.

chunk_size=32 splits 97.9% of documents: aggressive fine-grained chunking,
forces nearly every document to be split into multiple small pieces.

### Final chunk size selection: 32, 128, 256

- 32: fine grained, high splitting rate, precise but potentially loses context
- 128: balanced, aligns with corpus median, standard RAG reference point
- 256: coarse, minimal splitting, large context per chunk
### Why chunk_size=128 is used for the top-k analysis
When studying the effect of top-k (Analysis 1), we need to fix chunk size
to isolate the variable. chunk_size=128 is chosen because:
- It is one of the most commonly used default in RAG research
- It produces real chunking behavior on our corpus (41.1% of docs split)
- It sits at the corpus median -> neither too aggressive nor too coarse
- Results at this size are most comparable to existing RAG literature
